In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment
import glob
import os


# =============================
# Kalman Tracker
# =============================
class KalmanTracker: 
    def __init__(self, x, y, id):
        self.id = id
        self.state = np.array([x, y, 0, 0], dtype=float)

        self.P = np.eye(4) * 100
        self.F = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]])
        self.H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
        self.R = np.eye(2) * 5
        self.Q = np.eye(4)

        self.missed = 0

    def predict(self):
        self.state = self.F @ self.state
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.state[:2]

    def update(self, z):
        z = np.array(z)

        y = z - self.H @ self.state
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)

        self.state = self.state + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P

        self.missed = 0

    def get_pos(self):
        return self.state[:2]


# =============================
# Tracker Manager
# =============================
class TrackerManager:
    def __init__(self):
        self.trackers = []
        self.next_id = 0

    def update(self, detections):
        predictions = []

        for trk in self.trackers:
            predictions.append(trk.predict())

        if len(predictions) == 0:
            for det in detections:
                self.trackers.append(KalmanTracker(det[0], det[1], self.next_id))
                self.next_id += 1
            return

        cost = np.zeros((len(predictions), len(detections)))

        for i, pred in enumerate(predictions):
            for j, det in enumerate(detections):
                cost[i, j] = np.linalg.norm(pred - det)

        row_ind, col_ind = linear_sum_assignment(cost)

        assigned_trk = set()
        assigned_det = set()

        for r, c in zip(row_ind, col_ind):
            MAX_DISTANCE = 30  # tighter

            if cost[r, c] < MAX_DISTANCE:
                # if cost[r, c] < 50:  # tune this
                self.trackers[r].update(detections[c])
                assigned_trk.add(r)
                assigned_det.add(c)

        # unmatched trackers
        for i, trk in enumerate(self.trackers):
            if i not in assigned_trk:
                trk.missed += 1

        # remove dead tracks
        self.trackers = [t for t in self.trackers if t.missed < 20]

        # new trackers
        for i, det in enumerate(detections):
            if i not in assigned_det:
                self.trackers.append(KalmanTracker(det[0], det[1], self.next_id))
                self.next_id += 1


# =============================
# Load model
# =============================
model = YOLO("runs/detect/train3/weights/best.pt")

# =============================
# Load frames
# =============================
frame_paths = sorted(
    glob.glob("G://Sahajan_sir(DATA)//5gm_NaBrO3_0.5MH2SO4//2.1/*.png")
)

if len(frame_paths) == 0:
    raise Exception("No frames found!")

# read first frame
sample = cv2.imread(frame_paths[0])
h, w = sample.shape[:2]

# video writer
out = cv2.VideoWriter(
    "kalman_tracking.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h)
)

tracker = TrackerManager()


# color function
def id_color(pid):
    np.random.seed(pid)
    return tuple(int(x) for x in np.random.randint(100, 255, 3))


# =============================
# MAIN LOOP
# =============================
for frame_idx, img_path in enumerate(frame_paths):
    frame = cv2.imread(img_path)
    if frame is None:
        continue

    # YOLO detection
    results = model(frame, conf=0.5)

    detections = []
    for box in results[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = box
        w_box = x2 - x1

        # optional filter
        if 10 < w_box < 80:
            x = (x1 + x2) / 2
            y = (y1 + y2) / 2
            detections.append([x, y])

    detections = np.array(detections)

    # update tracker
    tracker.update(detections)

    # draw
    for trk in tracker.trackers:
        x, y = trk.get_pos()
        pid = trk.id
        color = id_color(pid)

        cv2.circle(frame, (int(x), int(y)), 8, color, 2)
        cv2.putText(
            frame,
            f"id:{pid}",
            (int(x) + 5, int(y) - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            1,
        )

    out.write(frame)

    print(f"Frame {frame_idx}/{len(frame_paths)}", end="\r")

out.release()

print("\nSaved: kalman_tracking.mp4")



0: 1024x1024 (no detections), 115.8ms
Speed: 11.2ms preprocess, 115.8ms inference, 0.9ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 0/4026
0: 1024x1024 (no detections), 10.5ms
Speed: 8.7ms preprocess, 10.5ms inference, 0.7ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 1/4026
0: 1024x1024 (no detections), 10.9ms
Speed: 7.5ms preprocess, 10.9ms inference, 0.8ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 2/4026
0: 1024x1024 (no detections), 10.3ms
Speed: 7.9ms preprocess, 10.3ms inference, 0.7ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 3/4026
0: 1024x1024 (no detections), 10.1ms
Speed: 7.6ms preprocess, 10.1ms inference, 0.9ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 4/4026
0: 1024x1024 (no detections), 10.3ms
Speed: 7.7ms preprocess, 10.3ms inference, 0.7ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 5/4026
0: 1024x1024 (no detections), 11.5ms
Speed: 9.0ms preprocess, 11.5ms inference, 1.3ms postprocess per 

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment
import glob
import os


# =============================
# CONFIG
# =============================
OUTPUT_FOLDER = "output_frames"
START_INDEX = 380  # change this to skip frames
SAVE_CSV = True
CSV_FILE = "tracks.csv"  # change to .dat if needed
SAVE_VELOCITY = True  # optional


# =============================
# Kalman Tracker
# =============================
class KalmanTracker:
    def __init__(self, x, y, id):
        self.id = id
        self.state = np.array([x, y, 0, 0], dtype=float)

        self.P = np.eye(4) * 100
        self.F = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]])
        self.H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
        self.R = np.eye(2) * 5
        self.Q = np.eye(4)

        self.missed = 0

    def predict(self):
        self.state = self.F @ self.state
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.state[:2]

    def update(self, z):
        z = np.array(z)

        y = z - self.H @ self.state
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)

        self.state = self.state + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P

        self.missed = 0

    def get_pos(self):
        return self.state[:2]


# =============================
# Tracker Manager
# =============================
class TrackerManager:
    def __init__(self):
        self.trackers = []
        self.next_id = 0

    def update(self, detections):
        predictions = []

        for trk in self.trackers:
            predictions.append(trk.predict())

        if len(predictions) == 0:
            for det in detections:
                self.trackers.append(KalmanTracker(det[0], det[1], self.next_id))
                self.next_id += 1
            return

        cost = np.zeros((len(predictions), len(detections)))

        for i, pred in enumerate(predictions):
            for j, det in enumerate(detections):
                cost[i, j] = np.linalg.norm(pred - det)

        row_ind, col_ind = linear_sum_assignment(cost)

        assigned_trk = set()
        assigned_det = set()

        for r, c in zip(row_ind, col_ind):
            if cost[r, c] < 50:
                self.trackers[r].update(detections[c])
                assigned_trk.add(r)
                assigned_det.add(c)

        # unmatched trackers
        for i, trk in enumerate(self.trackers):
            if i not in assigned_trk:
                trk.missed += 1

        # remove dead tracks
        self.trackers = [t for t in self.trackers if t.missed < 20]

        # new trackers
        for i, det in enumerate(detections):
            if i not in assigned_det:
                self.trackers.append(KalmanTracker(det[0], det[1], self.next_id))
                self.next_id += 1


# =============================
# Load YOLO model
# =============================
model = YOLO("runs/detect/train3/weights/best.pt")


# =============================
# Load frames
# =============================
frame_paths = sorted(
    glob.glob("G://Sahajan_sir(DATA)//5gm_NaBrO3_0.5MH2SO4//2.1/*.png")
)

if len(frame_paths) == 0:
    raise Exception("No frames found!")

sample = cv2.imread(frame_paths[0])
h, w = sample.shape[:2]

# video writer
out = cv2.VideoWriter(
    "kalman_tracking.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h)
)

tracker = TrackerManager()

# create output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# open CSV
if SAVE_CSV:
    f_csv = open(CSV_FILE, "w")
    if SAVE_VELOCITY:
        f_csv.write("frame,id,x,y,vx,vy\n")
    else:
        f_csv.write("frame,id,x,y\n")


# =============================
# color function
# =============================
def id_color(pid):
    np.random.seed(pid)
    return tuple(int(x) for x in np.random.randint(100, 255, 3))


# =============================
# MAIN LOOP
# =============================
for frame_idx, img_path in enumerate(frame_paths[START_INDEX:], start=START_INDEX):
    frame = cv2.imread(img_path)
    if frame is None:
        continue

    # YOLO detection
    results = model(frame, conf=0.5)

    detections = []
    for box in results[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = box
        w_box = x2 - x1

        # filter small/large noise
        if 10 < w_box < 80:
            x = (x1 + x2) / 2
            y = (y1 + y2) / 2
            detections.append([x, y])

    detections = np.array(detections)

    # update tracker
    tracker.update(detections)

    # draw + save data
    for trk in tracker.trackers:
        x, y = trk.get_pos()
        pid = trk.id
        color = id_color(pid)

        cv2.circle(frame, (int(x), int(y)), 8, color, 2)
        cv2.putText(
            frame,
            f"id:{pid}",
            (int(x) + 5, int(y) - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            1,
        )

        # write CSV/DAT
        if SAVE_CSV:
            if SAVE_VELOCITY:
                vx, vy = trk.state[2], trk.state[3]
                f_csv.write(f"{frame_idx},{pid},{x:.2f},{y:.2f},{vx:.2f},{vy:.2f}\n")
            else:
                f_csv.write(f"{frame_idx},{pid},{x:.2f},{y:.2f}\n")

    # save frame image
    out_path = os.path.join(OUTPUT_FOLDER, f"frame_{frame_idx:05d}.png")
    cv2.imwrite(out_path, frame)

    # write video
    out.write(frame)

    print(f"Frame {frame_idx}/{len(frame_paths)}", end="\r")


# =============================
# CLEANUP
# =============================
out.release()

if SAVE_CSV:
    f_csv.close()

print("\nSaved video: kalman_tracking.mp4")
print(f"Saved frames in: {OUTPUT_FOLDER}")
print(f"Saved tracking data: {CSV_FILE}")



0: 1024x1024 6 droplets, 50.6ms
Speed: 12.0ms preprocess, 50.6ms inference, 1.8ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 380/4026
0: 1024x1024 6 droplets, 11.0ms
Speed: 8.8ms preprocess, 11.0ms inference, 1.8ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 381/4026
0: 1024x1024 6 droplets, 12.2ms
Speed: 10.0ms preprocess, 12.2ms inference, 2.9ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 382/4026
0: 1024x1024 6 droplets, 11.7ms
Speed: 10.1ms preprocess, 11.7ms inference, 1.9ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 383/4026
0: 1024x1024 6 droplets, 10.8ms
Speed: 10.1ms preprocess, 10.8ms inference, 1.9ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 384/4026
0: 1024x1024 6 droplets, 10.3ms
Speed: 8.7ms preprocess, 10.3ms inference, 2.0ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 385/4026
0: 1024x1024 6 droplets, 10.1ms
Speed: 8.8ms preprocess, 10.1ms inference, 2.2ms postprocess per image at shape (1, 3, 

In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment
import glob
import os


# =============================
# CONFIG
# =============================
OUTPUT_FOLDER = "output_frames"
START_INDEX = 380
CSV_FILE = "tracks.csv"

MAX_DISTANCE = 30  # spatial gating
MISSED_THRESHOLD = 40  # keep lost tracks longer
APPEARANCE_WEIGHT = 0.5  # tune this


# =============================
# Feature extractor
# =============================
def extract_feature(frame, box):
    x1, y1, x2, y2 = map(int, box)
    crop = frame[y1:y2, x1:x2]

    if crop.size == 0:
        return np.array([0, 0])

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    mean_intensity = np.mean(gray)
    area = (x2 - x1) * (y2 - y1)

    return np.array([mean_intensity, area])


# =============================
# Kalman Tracker
# =============================
class KalmanTracker:
    def __init__(self, x, y, feature, id):
        self.id = id
        self.state = np.array([x, y, 0, 0], dtype=float)

        self.P = np.eye(4) * 100
        self.F = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]])
        self.H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
        self.R = np.eye(2) * 5
        self.Q = np.eye(4)

        self.missed = 0
        self.appearance = feature

    def predict(self):
        self.state = self.F @ self.state
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.state[:2]

    def update(self, z, feature):
        z = np.array(z)

        y = z - self.H @ self.state
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)

        self.state = self.state + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P

        self.missed = 0

        # update appearance (moving average)
        if self.appearance is None:
            self.appearance = feature
        else:
            self.appearance = 0.8 * self.appearance + 0.2 * feature

    def get_pos(self):
        return self.state[:2]


# =============================
# Tracker Manager
# =============================
class TrackerManager:
    def __init__(self):
        self.trackers = []
        self.next_id = 0

    def update(self, detections):
        predictions = [trk.predict() for trk in self.trackers]

        if len(predictions) == 0:
            for det in detections:
                self.trackers.append(
                    KalmanTracker(det[0], det[1], det[2], self.next_id)
                )
                self.next_id += 1
            return

        cost = np.zeros((len(self.trackers), len(detections)))

        for i, trk in enumerate(self.trackers):
            for j, det in enumerate(detections):
                pos = det[:2]
                feat = det[2]

                # spatial distance
                spatial = np.linalg.norm(predictions[i] - pos)

                # appearance distance
                if trk.appearance is None:
                    appearance = 0
                else:
                    appearance = np.linalg.norm(trk.appearance - feat)

                cost[i, j] = spatial + APPEARANCE_WEIGHT * appearance

        row_ind, col_ind = linear_sum_assignment(cost)

        assigned_trk = set()
        assigned_det = set()

        for r, c in zip(row_ind, col_ind):
            if cost[r, c] < MAX_DISTANCE:
                self.trackers[r].update(detections[c][:2], detections[c][2])
                assigned_trk.add(r)
                assigned_det.add(c)

        # unmatched trackers
        for i, trk in enumerate(self.trackers):
            if i not in assigned_trk:
                trk.missed += 1

        # remove dead tracks
        self.trackers = [t for t in self.trackers if t.missed < MISSED_THRESHOLD]

        # new trackers
        for i, det in enumerate(detections):
            if i not in assigned_det:
                self.trackers.append(
                    KalmanTracker(det[0], det[1], det[2], self.next_id)
                )
                self.next_id += 1


# =============================
# Load YOLO
# =============================
model = YOLO("runs/detect/train3/weights/best.pt")


# =============================
# Load frames
# =============================
frame_paths = sorted(
    glob.glob("G://Sahajan_sir(DATA)//5gm_NaBrO3_0.5MH2SO4//2.1/*.png")
)

if len(frame_paths) == 0:
    raise Exception("No frames found!")

sample = cv2.imread(frame_paths[0])
h, w = sample.shape[:2]

out = cv2.VideoWriter(
    "kalman_tracking.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h)
)

tracker = TrackerManager()
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# CSV
f_csv = open(CSV_FILE, "w")
f_csv.write("frame,id,x,y,vx,vy\n")


# =============================
# Color function
# =============================
def id_color(pid):
    np.random.seed(pid)
    return tuple(int(x) for x in np.random.randint(100, 255, 3))


# =============================
# MAIN LOOP
# =============================
for frame_idx, img_path in enumerate(frame_paths[START_INDEX:], start=START_INDEX):
    frame = cv2.imread(img_path)
    if frame is None:
        continue

    results = model(frame, conf=0.5)

    detections = []
    for box in results[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = box
        w_box = x2 - x1

        if 10 < w_box < 80:
            x = (x1 + x2) / 2
            y = (y1 + y2) / 2

            feature = extract_feature(frame, box)
            detections.append([x, y, feature])

    # update tracker
    tracker.update(detections)

    # draw + save
    for trk in tracker.trackers:
        x, y = trk.get_pos()
        vx, vy = trk.state[2], trk.state[3]
        pid = trk.id
        color = id_color(pid)

        cv2.circle(frame, (int(x), int(y)), 8, color, 2)
        cv2.putText(
            frame,
            f"id:{pid}",
            (int(x) + 5, int(y) - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            1,
        )

        # save data
        f_csv.write(f"{frame_idx},{pid},{x:.2f},{y:.2f},{vx:.2f},{vy:.2f}\n")

    # save frame
    out_path = os.path.join(OUTPUT_FOLDER, f"frame_{frame_idx:05d}.png")
    cv2.imwrite(out_path, frame)

    out.write(frame)

    print(f"Frame {frame_idx}/{len(frame_paths)}", end="\r")


# =============================
# CLEANUP
# =============================
out.release()
f_csv.close()

print("\nSaved video: kalman_tracking.mp4")
print(f"Saved frames in: {OUTPUT_FOLDER}")
print(f"Saved tracking data: {CSV_FILE}")



0: 1024x1024 6 droplets, 13.0ms
Speed: 20.0ms preprocess, 13.0ms inference, 13.2ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 380/4026
0: 1024x1024 6 droplets, 11.8ms
Speed: 8.6ms preprocess, 11.8ms inference, 1.9ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 381/4026
0: 1024x1024 6 droplets, 11.8ms
Speed: 8.6ms preprocess, 11.8ms inference, 2.1ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 382/4026
0: 1024x1024 6 droplets, 10.7ms
Speed: 8.2ms preprocess, 10.7ms inference, 2.0ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 383/4026
0: 1024x1024 6 droplets, 11.4ms
Speed: 8.3ms preprocess, 11.4ms inference, 1.7ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 384/4026
0: 1024x1024 6 droplets, 10.1ms
Speed: 8.2ms preprocess, 10.1ms inference, 2.0ms postprocess per image at shape (1, 3, 1024, 1024)
Frame 385/4026
0: 1024x1024 6 droplets, 10.7ms
Speed: 8.5ms preprocess, 10.7ms inference, 2.1ms postprocess per image at shape (1, 3, 10